# ray.shutdown()

<font color='red'>ray.shutdown() 是 Ray 框架中用于断开当前客户端与 Ray 集群的连接，并清理相关资源的核心 API。</font>它通常与 ray.init() 成对出现，标志着一个 Ray 任务或会话的结束。

## 🎯 核心功能

当你调用 ray.shutdown() 时，Ray 会在后台执行一系列清理操作，主要包括：
1. <font color='red'>断开连接：</font>断开当前 Driver（主程序）或 Worker（工作进程）与 Ray 集群（GCS 和 Raylet）的连接。
2. <font color='red'>清理全局状态：</font>清理当前 Job 注册的所有对象（Object）、远程函数（Remote Function）定义和 Actor 实例。这意味着你将无法再通过 ObjectRef 获取数据，也无法调用之前定义的远程函数或 Actor。
3. <font color='red'>终止进程：</font>如果当前是在本地模式（即通过 ray.init() 启动了一个本地临时集群），它会终止由该命令启动的后台进程（如 Raylet, GCS, Object Store 等）。

## 🚀 常见使用场景

###  脚本结束时的资源释放

在编写独立的 Python 脚本时，为了防止脚本运行结束后残留进程占用端口或内存，通常会在脚本末尾（或 finally 块中）调用它。

In [ ]:
import ray

ray.init()
# ... 执行分布式任务 ...
ray.shutdown() # 确保资源被释放

### 自动化测试与开发调试

<font color='red'>在单元测试或频繁重启的开发环境中，为了避免“端口已被占用”（Address already in use）的错误，通常会在 ray.init() 之前先尝试关闭旧的连接，或者利用 ignore_reinit_error=True 参数。</font>

In [ ]:
import ray

# 模式 A：手动清理
if ray.is_initialized():
    ray.shutdown()
ray.init(num_cpus=4)

# 模式 B：使用参数忽略重连错误（推荐用于测试）
ray.init(ignore_reinit_error=True)

### 集成环境（如 Azure Databricks）

在云原生环境或 Notebook 中，Ray 集群的生命周期通常由平台管理（例如作业完成后自动关闭）。但在某些情况下，你可能需要显式调用 ray.shutdown() 来提前释放资源，或者配合特定的集群管理库（如 shutdown_ray_cluster）使用。

## ⚠️ 关键注意事项
- 不可逆性：一旦调用 ray.shutdown()，当前 Python 进程中的 Ray 环境就被销毁了。如果你想继续使用 Ray，必须再次调用 ray.init() 重新初始化。
- <b>作用域限制：ray.shutdown() 仅断开当前客户端的连接。</b>
    - <font color='red'>如果你连接的是一个现有的远程集群（例如通过 ray.init(address="...")），调用 shutdown() 不会 关闭整个集群，只会断开你当前脚本的连接，集群和其他连接的用户不受影响。</font>
    - <font color='red'>如果你是启动本地集群的人（例如直接运行 ray.init()），调用 shutdown() 会关闭这个本地集群。</font>
- 与 ray.kill() 的区别：
    - ray.shutdown() 是针对当前 Driver/客户端的，用于退出环境。
    - ray.kill(actor) 是针对特定 Actor 的，用于强制终止某个具体的 Actor 实例。
    
    
| 特性 | `ray.init()` | `ray.shutdown()` |
| :--- | :--- | :--- |
| 功能 | 启动或连接集群 | 断开连接并清理 |
| 资源影响 | 分配资源，启动进程 | 释放资源，终止进程 |
| 调用时机 | 任务开始时 | 任务结束或需要重置时 |
| 状态 | 建立全局状态 | 清除全局状态 |

